In [1]:
import chromadb, pandas as pd, sentence_transformers as st
import re, json

/Users/infibiss/Desktop/Fridge-Recipe-Detector/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
df1 = pd.read_csv('recipes_foodru_1.csv')
df1.head()

,id,title,url,recipe,time,count,ingredients,nutrients,allergy,category
0,0,Шулюм из свинины,https://food.ru/recipes/256847-shulium-iz-svininy,Ошибка,0,0.0,[],{},Ошибка,первые блюда
1,1,Окрошка с крабом на айране,https://food.ru/recipes/254067-okroshka-s-krab...,Ошибка,0,0.0,[],{},Ошибка,первые блюда
2,2,Окрошка с отварной говядиной на ряженке,https://food.ru/recipes/254072-okroshka-s-otva...,Ошибка,0,0.0,[],{},Ошибка,первые блюда
3,3,Окрошка с копченой курицей и нутом на белом квасе,https://food.ru/recipes/254135-okroshka-s-kopc...,Ошибка,0,0.0,[],{},Ошибка,первые блюда
4,4,Окрошка со щавелем и ветчиной на кефире,https://food.ru/recipes/254063-okroshka-so-shc...,Ошибка,0,0.0,[],{},Ошибка,первые блюда


## Обработка

In [3]:
df1 = df1[df1['time'] != 0]
df1.head()

,id,title,url,recipe,time,count,ingredients,nutrients,allergy,category
201,201,Сырный суп со скумбрией,https://food.ru/recipes/227818-syrnyi-sup-so-s...,Нарежьте бекон на небольшие кусочки. Разогрейт...,30,4.0,"[{'name': 'Консервированная скумбрия', 'qty': ...","{'Калории': '202,74', 'Белки': '12,75', 'Жиры'...","Белок коровьего молока, Ракообразные, Рыба",первые блюда
202,202,Густой суп из шампиньонов с сыром и брокколи,https://food.ru/recipes/176803-sup-iz-gribov-s...,Нарежьте шампиньоны пластинками. Обжарьте в ма...,30,3.0,"[{'name': 'Шампиньоны', 'qty': '4', 'grams': '...","{'Калории': '28,68', 'Белки': '1,33', 'Жиры': ...","Белок коровьего молока, Злаки, содержащие глютен",первые блюда
203,203,Как сварить любой суп,https://food.ru/recipes/197715-kak-svarit-liub...,"Положите в кастрюлю кусок мяса, залейте водой ...",100,4.0,"[{'name': 'Свинина на кости', 'qty': '300', 'g...","{'Калории': '165,85', 'Белки': '7,59', 'Жиры':...","Злаки, содержащие глютен",первые блюда
204,204,Суп из сома,https://food.ru/recipes/228166-sup-iz-soma,Положите стейки сома в кастрюлю. Вскипятите в ...,40,3.0,"[{'name': 'Сом морской', 'qty': '300', 'grams'...","{'Калории': '33,69', 'Белки': '2,12', 'Жиры': ...","Злаки, содержащие глютен, Рыба",первые блюда
205,205,Татарская лапша,https://food.ru/recipes/228087-tatarskaja-lapsha,"Взбейте куриные яйца с 0,5 ч.л. соли при помощ...",180,4.0,"[{'name': 'Пшеничная мука хлебопекарная', 'qty...","{'Калории': '55,03', 'Белки': '3,73', 'Жиры': ...","Злаки, содержащие глютен, Яйцо",первые блюда


In [4]:
df2 = pd.read_csv('recipes_foodru_2.csv')
df2_valid = df2[df2['time'] != 0].copy()
df2_valid.head()

,id,title,url,recipe,time,count,ingredients,nutrients,allergy,category
0,0,Гнезда с тефтелями в томатном соусе,https://food.ru/recipes/257868-gnezda-s-teftel...,Крупно натрите морковь. Нарежьте лук кубиками ...,60,4.0,"[{'name': 'Фарш из свинины и говядины', 'qty':...","{'Калории': '163,32', 'Белки': '6,66', 'Жиры':...","Белок коровьего молока, Злаки, содержащие глютен",вторые блюда
1,1,Тушеные баклажаны с фасолью,https://food.ru/recipes/257915-baklazhany-s-fa...,Отварите предварительно замоченную фасоль до г...,75,2.0,"[{'name': 'Баклажан', 'qty': '1', 'grams': '21...","{'Калории': '128,74', 'Белки': '4,77', 'Жиры':...",Кунжут,вторые блюда
2,2,Cалат из зеленого горошка с сырокопченым карпаччо,https://food.ru/recipes/258025-calat-iz-zeleno...,Нашинкуйте капусту полосками шириной 5–6 мм. Р...,10,4.0,"[{'name': 'Консервированный горошек', 'qty': '...","{'Калории': '55,05', 'Белки': '3,86', 'Жиры': ...","Горчица, Пищевые добавки",вторые блюда
3,3,Легкий салат с кукурузой,https://food.ru/recipes/255758-legkii-salat-s-...,"Положите яйца в кастрюлю, залейте их водой. По...",25,4.0,"[{'name': 'Салат–латук', 'qty': '1', 'grams': ...","{'Калории': '64,5', 'Белки': '3,69', 'Жиры': '...","Белок коровьего молока, Яйцо",вторые блюда
4,4,Фетучини карбонара,https://food.ru/recipes/257663-fetuchini-karbo...,Отварите фетучини в кипящей подсоленной воде д...,20,4.0,"[{'name': 'Паста фетучини', 'qty': '400', 'gra...","{'Калории': '410,84', 'Белки': '20,5', 'Жиры':...","Белок коровьего молока, Злаки, содержащие глют...",вторые блюда


In [5]:
# df3 = pd.read_csv('recipes_foodru_вторые блюда.csv')
# df3_valid = df3[df3['time'] != 0].copy()
# df3_valid.head()

FileNotFoundError: [Errno 2] No such file or directory: 'recipes_foodru_вторые блюда.csv'

In [33]:
# combined_df = pd.concat([df3_valid, df2_valid], ignore_index=True)
# final_df = combined_df.drop_duplicates(subset=['id'], keep='first')
# final_df.head(50)

,id,title,url,recipe,time,count,ingredients,nutrients,allergy,category
0,0,Гнезда с тефтелями в томатном соусе,https://food.ru/recipes/257868-gnezda-s-teftel...,Крупно натрите морковь. Нарежьте лук кубиками ...,60,4.0,"[{'name': 'Фарш из свинины и говядины', 'qty':...","{'Калории': '163,32', 'Белки': '6,66', 'Жиры':...","Белок коровьего молока, Злаки, содержащие глютен",вторые блюда
1,1,Тушеные баклажаны с фасолью,https://food.ru/recipes/257915-baklazhany-s-fa...,Отварите предварительно замоченную фасоль до г...,75,2.0,"[{'name': 'Баклажан', 'qty': '1', 'grams': '21...","{'Калории': '128,74', 'Белки': '4,77', 'Жиры':...",Кунжут,вторые блюда
2,2,Cалат из зеленого горошка с сырокопченым карпаччо,https://food.ru/recipes/258025-calat-iz-zeleno...,Нашинкуйте капусту полосками шириной 5–6 мм. Р...,10,4.0,"[{'name': 'Консервированный горошек', 'qty': '...","{'Калории': '55,05', 'Белки': '3,86', 'Жиры': ...","Горчица, Пищевые добавки",вторые блюда
3,3,Легкий салат с кукурузой,https://food.ru/recipes/255758-legkii-salat-s-...,"Положите яйца в кастрюлю, залейте их водой. По...",25,4.0,"[{'name': 'Салат–латук', 'qty': '1', 'grams': ...","{'Калории': '64,5', 'Белки': '3,69', 'Жиры': '...","Белок коровьего молока, Яйцо",вторые блюда
4,4,Фетучини карбонара,https://food.ru/recipes/257663-fetuchini-karbo...,Отварите фетучини в кипящей подсоленной воде д...,20,4.0,"[{'name': 'Паста фетучини', 'qty': '400', 'gra...","{'Калории': '410,84', 'Белки': '20,5', 'Жиры':...","Белок коровьего молока, Злаки, содержащие глют...",вторые блюда
5,5,Салат «Ацецили» с курицей,https://food.ru/recipes/255756-salat-acecili-s...,"Нарежьте дайкон соломкой, а болгарский перец —...",15,4.0,"[{'name': 'Куриная грудка', 'qty': '180', 'gra...","{'Калории': '51,76', 'Белки': '6,71', 'Жиры': ...","Белок коровьего молока, Горчица",вторые блюда
6,6,Шоколадные вареники с творогом и вишней,https://food.ru/recipes/257266-shokoladnye-var...,Всыпьте в миску с мукой какао-порошок и соль. ...,35,4.0,"[{'name': 'Пшеничная мука хлебопекарная', 'qty...","{'Калории': '182,28', 'Белки': '7,33', 'Жиры':...","Белок коровьего молока, Злаки, содержащие глют...",вторые блюда
7,7,Простой блинный торт с черникой,https://food.ru/recipes/256678-blinchiki-s-che...,Всыпьте сахар в молоко и разбейте туда же яйца...,70,6.0,"[{'name': 'Молоко', 'qty': '2.5', 'grams': '50...","{'Калории': '155,06', 'Белки': '4,64', 'Жиры':...","Белок коровьего молока, Злаки, содержащие глют...",вторые блюда
8,8,Зеленый салат с семенами и орехами и мини-моца...,https://food.ru/recipes/255754-zelenyi-salat-s...,Порвите салатные листья руками на небольшие ку...,15,4.0,"[{'name': 'Салат–латук', 'qty': '1', 'grams': ...","{'Калории': '113,72', 'Белки': '5,41', 'Жиры':...","Белок коровьего молока, Кунжут",вторые блюда
9,9,Вареники с картошкой и грибами,https://food.ru/recipes/257256-Vareniki-s-kart...,"Всыпьте в миску с мукой 1 ч.л. соли, перемешай...",60,3.0,"[{'name': 'Пшеничная мука хлебопекарная', 'qty...","{'Калории': '155,61', 'Белки': '4,68', 'Жиры':...","Белок коровьего молока, Злаки, содержащие глют...",вторые блюда


In [44]:
# final_df.to_csv('recipes_foodru_2.csv', index=False, encoding='utf-8-sig')

Убираем нерабочее

In [37]:
# df1_no_id = df1.reset_index(drop=True)
# df_final_no_id = final_df.reset_index(drop=True)
# 
# df = pd.concat([df1_no_id, df_final_no_id], ignore_index=True)
# df['id'] = df.index + 1

In [6]:
df = pd.read_csv('recipes_foodru.csv')
df.head(100)

,id,title,url,recipe,time,count,ingredients,nutrients,allergy,category
0,1,Сырный суп со скумбрией,https://food.ru/recipes/227818-syrnyi-sup-so-s...,Нарежьте бекон на небольшие кусочки. Разогрейт...,30,4.0,"[{'name': 'Консервированная скумбрия', 'qty': ...","{'Калории': '202,74', 'Белки': '12,75', 'Жиры'...","Белок коровьего молока, Ракообразные, Рыба",первые блюда
1,2,Густой суп из шампиньонов с сыром и брокколи,https://food.ru/recipes/176803-sup-iz-gribov-s...,Нарежьте шампиньоны пластинками. Обжарьте в ма...,30,3.0,"[{'name': 'Шампиньоны', 'qty': '4', 'grams': '...","{'Калории': '28,68', 'Белки': '1,33', 'Жиры': ...","Белок коровьего молока, Злаки, содержащие глютен",первые блюда
2,3,Как сварить любой суп,https://food.ru/recipes/197715-kak-svarit-liub...,"Положите в кастрюлю кусок мяса, залейте водой ...",100,4.0,"[{'name': 'Свинина на кости', 'qty': '300', 'g...","{'Калории': '165,85', 'Белки': '7,59', 'Жиры':...","Злаки, содержащие глютен",первые блюда
3,4,Суп из сома,https://food.ru/recipes/228166-sup-iz-soma,Положите стейки сома в кастрюлю. Вскипятите в ...,40,3.0,"[{'name': 'Сом морской', 'qty': '300', 'grams'...","{'Калории': '33,69', 'Белки': '2,12', 'Жиры': ...","Злаки, содержащие глютен, Рыба",первые блюда
4,5,Татарская лапша,https://food.ru/recipes/228087-tatarskaja-lapsha,"Взбейте куриные яйца с 0,5 ч.л. соли при помощ...",180,4.0,"[{'name': 'Пшеничная мука хлебопекарная', 'qty...","{'Калории': '55,03', 'Белки': '3,73', 'Жиры': ...","Злаки, содержащие глютен, Яйцо",первые блюда
...,...,...,...,...,...,...,...,...,...,...
95,96,Зеленые щи со щавелем,https://food.ru/recipes/214572-zelnye-shchi-s-...,Положите говядину в кастрюлю. Залейте водой и ...,130,4.0,"[{'name': 'Щавель', 'qty': '300', 'grams': '30...","{'Калории': '50,53', 'Белки': '1,77', 'Жиры': ...","Белок коровьего молока, Яйцо",первые блюда
96,97,Сырный суп с овощами,https://food.ru/recipes/213865-ovoshchnoi-syrn...,Нарежьте картофель и морковь кубиками со сторо...,45,3.0,"[{'name': 'Плавленый сыр', 'qty': '100', 'gram...","{'Калории': '30,2', 'Белки': '1,6', 'Жиры': '0...",Белок коровьего молока,первые блюда
97,98,Пшенный суп с курицей,https://food.ru/recipes/213864-pshennyi-sup-s-...,"Положите куриный окорочок в кастрюлю, залейте ...",50,3.0,"[{'name': 'Куриный окорочок', 'qty': '300', 'g...","{'Калории': '44,21', 'Белки': '2,78', 'Жиры': ...",Нет,первые блюда
98,99,Суп с ячневой крупой,https://food.ru/recipes/213814-sup-s-jachnevoi...,Разрежьте куриные крылья на несколько частей. ...,75,4.0,"[{'name': 'Вода', 'qty': '2', 'grams': '2000'}...","{'Калории': '37,14', 'Белки': '2,45', 'Жиры': ...","Белок коровьего молока, Злаки, содержащие глютен",первые блюда


In [7]:
df.tail(20)

,id,title,url,recipe,time,count,ingredients,nutrients,allergy,category
4801,4802,Треска с картофелем и розмарином,https://food.ru/recipes/231064-treska-s-kartof...,Нарежьте картофель небольшими дольками. Положи...,45,3.0,"[{'name': 'Филе трески', 'qty': '600', 'grams'...","{'Калории': '98,21', 'Белки': '8,1', 'Жиры': '...","Белок коровьего молока, Горчица, Рыба",вторые блюда
4802,4803,Курица по-тоскански,https://food.ru/recipes/231141-Kurica-po-toska...,"Разрежьте каждое куриное филе на две половины,...",45,6.0,"[{'name': 'Куриное филе', 'qty': '500', 'grams...","{'Калории': '108,21', 'Белки': '9,43', 'Жиры':...","Белок коровьего молока, Пищевые добавки",вторые блюда
4803,4804,Пикантная сливочная лапша с креветками,https://food.ru/recipes/231045-pikantnaja-sliv...,Выложите креветки в миску. Добавьте к ним 1 ст...,60,2.0,"[{'name': 'Креветки', 'qty': '500', 'grams': '...","{'Калории': '185,05', 'Белки': '13,83', 'Жиры'...","Белок коровьего молока, Злаки, содержащие глют...",вторые блюда
4804,4805,Плов с яблоками,https://food.ru/recipes/231362-plov-s-jablokami,"Влейте в кастрюлю 1,25 стакана воды и всыпьте ...",70,4.0,"[{'name': 'Рис', 'qty': '90', 'grams': '90'}, ...","{'Калории': '116,59', 'Белки': '1,22', 'Жиры':...",Орехи,вторые блюда
4805,4806,Рыбные котлеты с зеленым пюре,https://food.ru/recipes/231103-rybnye-kotlety-...,Вскипятите в кастрюле воду с веточкой мяты. По...,25,2.0,"[{'name': 'Горох', 'qty': '1.5', 'grams': '307...","{'Калории': '42,22', 'Белки': '4,98', 'Жиры': ...","Белок коровьего молока, Рыба",вторые блюда
4806,4807,Индийское карри из хека,https://food.ru/recipes/231102-karri-iz-heka,"Снимите цедру с лаймов, измельчите ее. Разрежь...",120,4.0,"[{'name': 'Филе хека', 'qty': '600', 'grams': ...","{'Калории': '118,96', 'Белки': '7,74', 'Жиры':...","Пищевые добавки, Рыба",вторые блюда
4807,4808,Запеканка из овсянки с тыквой,https://food.ru/recipes/230665-ovsjanka-s-tykvoi,Нарежьте тыкву на небольшие кусочки и положите...,60,4.0,"[{'name': 'Тыква', 'qty': '250', 'grams': '250...","{'Калории': '144,34', 'Белки': '4,85', 'Жиры':...","Белок коровьего молока, Злаки, содержащие глют...",вторые блюда
4808,4809,Морковные драники,https://food.ru/recipes/230589-morkovnye-draniki,"Натрите морковь на средней терке, пармезан — н...",40,3.0,"[{'name': 'Морковь', 'qty': '250', 'grams': '2...","{'Калории': '190,5', 'Белки': '6,78', 'Жиры': ...","Белок коровьего молока, Злаки, содержащие глют...",вторые блюда
4809,4810,Ньокки из картофеля,https://food.ru/recipes/230530-nki,"Положите картофель в кастрюлю, залейте водой. ...",50,2.0,"[{'name': 'Картошка', 'qty': '2', 'grams': '24...","{'Калории': '179,68', 'Белки': '8,99', 'Жиры':...","Белок коровьего молока, Злаки, содержащие глют...",вторые блюда
4810,4811,Наггетсы из куриной грудки,https://food.ru/recipes/230528-naggetsy-iz-kur...,Хорошенько отбейте куриное филе с двух сторон ...,110,2.0,"[{'name': 'Куриное филе', 'qty': '450', 'grams...","{'Калории': '175,42', 'Белки': '18,22', 'Жиры'...","Белок коровьего молока, Злаки, содержащие глютен",вторые блюда


In [42]:
df.to_csv('recipes_foodru.csv', index=False, encoding='utf-8-sig')

Код обработки для эмбеддингов

In [4]:
# from nltk.corpus import stopwords, wordnet
# from nltk.stem import WordNetLemmatizer
# import nltk
# import warnings
# 
# warnings.filterwarnings('ignore')
# 
# # Скачиваем словари при первом запуске
# nltk.download('punkt', quiet=True)
# nltk.download('stopwords', quiet=True)
# nltk.download('wordnet', quiet=True)
# 
# STOPWORDS = set(stopwords.words('english'))
# lemmatizer = WordNetLemmatizer()
# 
# 
# # Функция для определения части речи
# def get_wordnet_pos(word):
#     tag = nltk.pos_tag([word])[0][1][0].upper()
#     tag_dict = {
#         'J': wordnet.ADJ,
#         'N': wordnet.NOUN,
#         'V': wordnet.VERB,
#         'R': wordnet.ADV
#     }
#     return tag_dict.get(tag, wordnet.NOUN)
# 
# 
# # Предобработка запроса теми же шагами
# def preprocess_query(q: str):
#     # В нижний шрифт
#     text = q.lower()
#     # Убираем markdown и ссылки
#     text = re.sub(r'[^a-z0-9\u0400-\u04FF\s]', ' ', text)
#     # Преобразуем в начальную форму
#     tokens = [lemmatizer.lemmatize(tok, get_wordnet_pos(tok)) for tok in text.split() if tok not in STOPWORDS and len(tok) > 2]
#     return ' '.join(tokens), tokens

In [8]:
import spacy

nlp = spacy.load("ru_core_news_md")

def preprocess_query(text: str):
    doc = nlp(text.lower())
    tokens = [
        token.lemma_ 
        for token in doc 
        if not token.is_stop and not token.is_punct and len(token.text) > 2
    ]
    return ' '.join(tokens), tokens

In [9]:
print(preprocess_query('Куриное филе с помидорами и специями.'))

('куриный филе помидор специя', ['куриный', 'филе', 'помидор', 'специя'])


Работаем с ингредиентами

In [10]:
from ast import literal_eval
df['ingredients'] = df['ingredients'].apply(literal_eval)

In [11]:
# df['ingredients_names_eng'] = 

In [12]:
df['ingredients_per_one'] = df.apply(
    lambda row: [
        {
            'name': preprocess_query(item['name'])[0],
            'qty': round(float((item.get('qty') or '0').replace(',', '.')) / row['count'], 2),
            'grams': round(float((item.get('grams') or '0').replace(',', '.')) / row['count'])
        }
        for item in row['ingredients']
    ],
    axis=1
)
df.head()

,id,title,url,recipe,time,count,ingredients,nutrients,allergy,category,ingredients_per_one
0,1,Сырный суп со скумбрией,https://food.ru/recipes/227818-syrnyi-sup-so-s...,Нарежьте бекон на небольшие кусочки. Разогрейт...,30,4.0,"[{'name': 'Консервированная скумбрия', 'qty': ...","{'Калории': '202,74', 'Белки': '12,75', 'Жиры'...","Белок коровьего молока, Ракообразные, Рыба",первые блюда,"[{'name': 'консервированный скумбрия', 'qty': ..."
1,2,Густой суп из шампиньонов с сыром и брокколи,https://food.ru/recipes/176803-sup-iz-gribov-s...,Нарежьте шампиньоны пластинками. Обжарьте в ма...,30,3.0,"[{'name': 'Шампиньоны', 'qty': '4', 'grams': '...","{'Калории': '28,68', 'Белки': '1,33', 'Жиры': ...","Белок коровьего молока, Злаки, содержащие глютен",первые блюда,"[{'name': 'шампиньон', 'qty': 1.33, 'grams': 4..."
2,3,Как сварить любой суп,https://food.ru/recipes/197715-kak-svarit-liub...,"Положите в кастрюлю кусок мяса, залейте водой ...",100,4.0,"[{'name': 'Свинина на кости', 'qty': '300', 'g...","{'Калории': '165,85', 'Белки': '7,59', 'Жиры':...","Злаки, содержащие глютен",первые блюда,"[{'name': 'свинина кость', 'qty': 75.0, 'grams..."
3,4,Суп из сома,https://food.ru/recipes/228166-sup-iz-soma,Положите стейки сома в кастрюлю. Вскипятите в ...,40,3.0,"[{'name': 'Сом морской', 'qty': '300', 'grams'...","{'Калории': '33,69', 'Белки': '2,12', 'Жиры': ...","Злаки, содержащие глютен, Рыба",первые блюда,"[{'name': 'сом морской', 'qty': 100.0, 'grams'..."
4,5,Татарская лапша,https://food.ru/recipes/228087-tatarskaja-lapsha,"Взбейте куриные яйца с 0,5 ч.л. соли при помощ...",180,4.0,"[{'name': 'Пшеничная мука хлебопекарная', 'qty...","{'Калории': '55,03', 'Белки': '3,73', 'Жиры': ...","Злаки, содержащие глютен, Яйцо",первые блюда,"[{'name': 'пшеничная мука хлебопекарный', 'qty..."


In [13]:
df['ingredients_names'] = df['ingredients'].apply(
    lambda items: "; ".join(preprocess_query(item['name'])[0] for item in items)
)
df.head()

,id,title,url,recipe,time,count,ingredients,nutrients,allergy,category,ingredients_per_one,ingredients_names
0,1,Сырный суп со скумбрией,https://food.ru/recipes/227818-syrnyi-sup-so-s...,Нарежьте бекон на небольшие кусочки. Разогрейт...,30,4.0,"[{'name': 'Консервированная скумбрия', 'qty': ...","{'Калории': '202,74', 'Белки': '12,75', 'Жиры'...","Белок коровьего молока, Ракообразные, Рыба",первые блюда,"[{'name': 'консервированный скумбрия', 'qty': ...",консервированный скумбрия; сладкий перец; поми...
1,2,Густой суп из шампиньонов с сыром и брокколи,https://food.ru/recipes/176803-sup-iz-gribov-s...,Нарежьте шампиньоны пластинками. Обжарьте в ма...,30,3.0,"[{'name': 'Шампиньоны', 'qty': '4', 'grams': '...","{'Калории': '28,68', 'Белки': '1,33', 'Жиры': ...","Белок коровьего молока, Злаки, содержащие глютен",первые блюда,"[{'name': 'шампиньон', 'qty': 1.33, 'grams': 4...",шампиньон; картошка; плавленый сыр; морковь; б...
2,3,Как сварить любой суп,https://food.ru/recipes/197715-kak-svarit-liub...,"Положите в кастрюлю кусок мяса, залейте водой ...",100,4.0,"[{'name': 'Свинина на кости', 'qty': '300', 'g...","{'Калории': '165,85', 'Белки': '7,59', 'Жиры':...","Злаки, содержащие глютен",первые блюда,"[{'name': 'свинина кость', 'qty': 75.0, 'grams...",свинина кость; соль; морковь; растительный мас...
3,4,Суп из сома,https://food.ru/recipes/228166-sup-iz-soma,Положите стейки сома в кастрюлю. Вскипятите в ...,40,3.0,"[{'name': 'Сом морской', 'qty': '300', 'grams'...","{'Калории': '33,69', 'Белки': '2,12', 'Жиры': ...","Злаки, содержащие глютен, Рыба",первые блюда,"[{'name': 'сом морской', 'qty': 100.0, 'grams'...",сом морской; картошка; репчатый лук; морковь; ...
4,5,Татарская лапша,https://food.ru/recipes/228087-tatarskaja-lapsha,"Взбейте куриные яйца с 0,5 ч.л. соли при помощ...",180,4.0,"[{'name': 'Пшеничная мука хлебопекарная', 'qty...","{'Калории': '55,03', 'Белки': '3,73', 'Жиры': ...","Злаки, содержащие глютен, Яйцо",первые блюда,"[{'name': 'пшеничная мука хлебопекарный', 'qty...",пшеничная мука хлебопекарный; куриный яйцо; со...


In [14]:
df['ingredients_names_grams'] = df['ingredients_per_one'].apply(
    lambda items: "; ".join(f"{preprocess_query(item['name'])[0]}: {item['grams']} г" for item in items)
)
df.head()

,id,title,url,recipe,time,count,ingredients,nutrients,allergy,category,ingredients_per_one,ingredients_names,ingredients_names_grams
0,1,Сырный суп со скумбрией,https://food.ru/recipes/227818-syrnyi-sup-so-s...,Нарежьте бекон на небольшие кусочки. Разогрейт...,30,4.0,"[{'name': 'Консервированная скумбрия', 'qty': ...","{'Калории': '202,74', 'Белки': '12,75', 'Жиры'...","Белок коровьего молока, Ракообразные, Рыба",первые блюда,"[{'name': 'консервированный скумбрия', 'qty': ...",консервированный скумбрия; сладкий перец; поми...,консервированный скумбрия: 60 г; сладкий перец...
1,2,Густой суп из шампиньонов с сыром и брокколи,https://food.ru/recipes/176803-sup-iz-gribov-s...,Нарежьте шампиньоны пластинками. Обжарьте в ма...,30,3.0,"[{'name': 'Шампиньоны', 'qty': '4', 'grams': '...","{'Калории': '28,68', 'Белки': '1,33', 'Жиры': ...","Белок коровьего молока, Злаки, содержащие глютен",первые блюда,"[{'name': 'шампиньон', 'qty': 1.33, 'grams': 4...",шампиньон; картошка; плавленый сыр; морковь; б...,шампиньон: 43 г; картошка: 40 г; плавленый сыр...
2,3,Как сварить любой суп,https://food.ru/recipes/197715-kak-svarit-liub...,"Положите в кастрюлю кусок мяса, залейте водой ...",100,4.0,"[{'name': 'Свинина на кости', 'qty': '300', 'g...","{'Калории': '165,85', 'Белки': '7,59', 'Жиры':...","Злаки, содержащие глютен",первые блюда,"[{'name': 'свинина кость', 'qty': 75.0, 'grams...",свинина кость; соль; морковь; растительный мас...,свинина кость: 75 г; соль: 0 г; морковь: 25 г;...
3,4,Суп из сома,https://food.ru/recipes/228166-sup-iz-soma,Положите стейки сома в кастрюлю. Вскипятите в ...,40,3.0,"[{'name': 'Сом морской', 'qty': '300', 'grams'...","{'Калории': '33,69', 'Белки': '2,12', 'Жиры': ...","Злаки, содержащие глютен, Рыба",первые блюда,"[{'name': 'сом морской', 'qty': 100.0, 'grams'...",сом морской; картошка; репчатый лук; морковь; ...,сом морской: 100 г; картошка: 80 г; репчатый л...
4,5,Татарская лапша,https://food.ru/recipes/228087-tatarskaja-lapsha,"Взбейте куриные яйца с 0,5 ч.л. соли при помощ...",180,4.0,"[{'name': 'Пшеничная мука хлебопекарная', 'qty...","{'Калории': '55,03', 'Белки': '3,73', 'Жиры': ...","Злаки, содержащие глютен, Яйцо",первые блюда,"[{'name': 'пшеничная мука хлебопекарный', 'qty...",пшеничная мука хлебопекарный; куриный яйцо; со...,пшеничная мука хлебопекарный: 49 г; куриный яй...


## Добавление в ChromaDB

для обновления, снести бд и рестартнуть ядро

In [15]:
model = st.SentenceTransformer('paraphrase-multilingual-mpnet-base-v2')

In [18]:
def add_to_chroma(df: pd.DataFrame, collection_name='chroma_db'):
    chroma = chromadb.PersistentClient(path=collection_name)
    
    # try:
    #     chroma.delete_collection(name=collection_name)
    # except ValueError:
    #     print(f"Коллекция '{collection_name}' не найдена, будет создана новая.")
        
        
    coll   = chroma.get_or_create_collection(
        name=collection_name,
        metadata={"hnsw:space": "cosine"}
    )

    docs       = df['ingredients_names'].tolist()
    embeds     = model.encode(docs, batch_size=64, show_progress_bar=True, normalize_embeddings=True)
    meta  = df[
        ['title', 'recipe', 'time', 'count', 'allergy', 'nutrients', 'ingredients', 'ingredients_per_one']
    ].to_dict(orient='records')
    
    for rec in meta:
        rec['ingredients'] = json.dumps(rec['ingredients'], ensure_ascii=False)
        rec['ingredients_per_one'] = json.dumps(rec['ingredients_per_one'], ensure_ascii=False)

    coll.add(
        documents = docs,
        ids       = df['id'].astype(str).tolist(),
        metadatas = meta,
        embeddings= embeds
    )
    
    print('Vectors in Chroma:', coll.count())

In [19]:
add_to_chroma(df, collection_name='chroma_db')

Batches: 100%|██████████| 76/76 [00:22<00:00,  3.34it/s]


Vectors in Chroma: 4821


## Поиск по базе

In [20]:
def embed_inventory(inv: dict[str,int]):
    text = "; ".join(preprocess_query(k)[0] for k, v in inv.items())
    return model.encode([text], normalize_embeddings=True)[0]

def retrieve_candidates(inv, topN=100, collection_name='chroma_db'):
    emb = embed_inventory(inv)
    chroma = chromadb.PersistentClient(path=collection_name)
    coll   = chroma.get_or_create_collection(collection_name)
    
    res = coll.query(
        query_embeddings=[emb.tolist()],
        n_results=topN,
        # where=where,
        include=['documents', 'metadatas', 'distances'],
    )
    return res

UNIT2G = {'г': 1, 'мл': 1, 'шт.': 60, 'стакан': 200, 'ст. л.': 18}
def coverage(recipe_ingrs: str, inv: list) -> float:
    items = json.loads(recipe_ingrs)
    got, total = 0, len(items)
    for ingr in items:
        # print(f"Checking ingredient: {ingr}")
        # по граммам
        # have = inv.get(name, 0) * UNIT2G.get('шт.',1)
        # if have >= need_g:
        #     got += 1
        
        # по названию
        name_cleaned = preprocess_query(ingr['name'])[0]
        # print(f"Checking {ingr['name']} in inventory...")
        if name_cleaned in inv:
            got += 1
    return got / total if total else 0

In [21]:
def top_k(inv: dict, k: int = 10, pool: int = 100):
    cand = retrieve_candidates(inv, topN=pool)
    print(f"Found {len(cand['documents'][0])}")
    
    # scored = []
    # inv_cleaned = [preprocess_query(k)[0] for k in inv.keys()]
    # print(f"Inventory cleaned: {inv_cleaned}")
    # for meta, doc in zip(cand['metadatas'][0], cand['documents'][0]):
    #     # print(f'Meta {meta}, doc {doc}')
    #     cov = coverage(meta['ingredients'], inv_cleaned)
    #     scored.append((cov, doc, meta))
    # scored.sort(key=lambda x: (-x[0], ))      # ↓ по coverage
    # return scored[:k]

    combined = zip(cand['distances'][0], cand['metadatas'][0])
    scored = sorted(combined, key=lambda x: x[0])

    return [meta for _, meta in scored[:k]]

Пример

In [25]:
inv = {
    'куриное филе': 2,
    'помидоры': 3,
    'лук': 3,
    'чеснок': 5,
    'оливковое масло': 1,
    'специи': 1
}

res = top_k(inv, k=3)
for meta in res:
    print(f"title: {meta['title']}, time: {meta['time']} мин")
    print(f"ingredients: {meta['ingredients']}")
    print(f"allergy: {meta['allergy']}")
    print(f"nutrients: {meta['nutrients']}")
    print()

Found 100
title: Аппетитное чахохбили с пряными куриными крылышками по-домашнему, time: 60 мин
ingredients: [{"name": "Куриные крылышки", "qty": "1500", "grams": "1500"}, {"name": "Репчатый лук", "qty": "300", "grams": "300"}, {"name": "Помидор", "qty": "320", "grams": "320"}, {"name": "Чеснок", "qty": "10", "grams": "10"}, {"name": "Соль", "qty": null, "grams": null}, {"name": "Кинза", "qty": null, "grams": null}, {"name": "Черный перец молотый", "qty": null, "grams": null}, {"name": "Сливочное масло", "qty": null, "grams": null}, {"name": "Томатный соус", "qty": "40", "grams": "40"}]
allergy: Белок коровьего молока
nutrients: {'Калории': '140,18', 'Белки': '13,69', 'Жиры': '8,61', 'Углеводы': '2,2'}

title: Сочное чахохбили из курицы с пряным соевым соусом, овощами и специями, time: 60 мин
ingredients: [{"name": "Курица", "qty": "1500", "grams": "1500"}, {"name": "Репчатый лук", "qty": "300", "grams": "300"}, {"name": "Помидор", "qty": "250", "grams": "250"}, {"name": "Соевый соус", 